In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv(r'C:\\Users\nmims.student\\Yearly_revenue.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\\\Users\\nmims.student\\\\Yearly_revenue.csv'

In [ ]:
Year = [2020 + i for i in range(len(df))]
df['Date'] = pd.to_datetime([f'{year}-01-01' for year in Year])

In [ ]:
df.head()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format = '%Y-%m-%d')
df = df.set_index('Date')

In [ ]:
df.head()

In [ ]:
sns.lineplot(df)
plt.ylabel('Yearly_revenue')

In [ ]:
result = seasonal_decompose(df[['Yearly_revenue']], model='multiplicative')
result.plot()
plt.show()

In [ ]:
pip install pymannkendall

In [ ]:
import pymannkendall as mk

#perform Mann-Kendall test
#H0: There is no monotonic trend in the series

mk.original_test(df['Yearly_revenue'])

In [ ]:
#p-value<0.5 reject H0 there is a trend present
train_df = df[:int(df.shape[0]*0.7)]
test_df = df[int(df.shape[0]*0.7):]

In [ ]:
train_df.shape

In [ ]:
#Double Exponential Smoothing

from statsmodels.tsa.api import Holt

model_double = Holt(train_df)
model_double_fit = model_double.fit()

In [ ]:
forecast_double = model_double_fit.forecast(6)
print(forecast_double)

In [ ]:
forecast_double = model_double_fit.forecast(40)
model_double_fit.params

In [ ]:
plt.plot(df,label='Original Data')
plt.plot(model_double_fit.fittedvalues, label='fitted Values')
plt.plot(forecast_double, label='Forecast')
plt.xlabel('Year')
plt.ylabel('Yearly Revenue')
plt.title('Double Exponential Smoothing')
plt.legend()
plt.show()

In [ ]:
#Practical-2 Checking Stationary
#ADF Test
#h0-series is not stationary vs h1-series is stationary

In [ ]:
from statsmodels.tsa.stattools import adfuller
print('Results of the Dickey Fuller Test:')
dftest = adfuller(df['Yearly_revenue'])
dfoutput=pd.Series(dftest[0:4],index=["Test Statistic",'P-value','#Lags used','Number of Observations'])
for key,value in dftest[4].items():
    dfoutput['Critical Value(%s)' %key] =value
print(dfoutput)

In [ ]:
#p-value>0.5 so we fail to reject the null hypothesis i.e Series is NOT stationary

In [ ]:
#KPSS Test
#h0-series is stationary vs h1-series is not stationary

In [ ]:
from statsmodels.tsa.stattools import kpss
kp = kpss(df['Yearly_revenue'])
p = kp[1]
print ("p-value for KPSS test (untransformed) = ", p)

In [ ]:
#p-value<0.5 so we fail to reject the null hypothesis i.e Series is NOT stationary

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.figure(figsize=(20,5))
plt.grid()
plot_acf(df['Yearly_revenue'], ax=plt.gca(), lags=30)
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
plt.grid()
plot_pacf(df['Yearly_revenue'], ax=plt.gca(), lags=20)
plt.show()

In [ ]:
#To make the data stationary we do differencing

In [ ]:
#First order non-seasonal differencing
diff = df['Yearly_revenue'].diff().dropna()
plt.figure(figsize=(14,3))
plt.grid()
plt.plot(diff)
plt.show()

In [ ]:
print('Results of the Dickey Fuller Test:')
dftest = adfuller(diff)
dfoutput=pd.Series(dftest[0:4],index=["Test Statistic",'P-value','#Lags used','Number of Observations'])
for key,value in dftest[4].items():
    dfoutput['Critical Value(%s)' %key] =value
print(dfoutput)
#it is not stationary

In [ ]:
kp = kpss(diff)
p = kp[1]
print ("p-value for KPSS test (untransformed) = ", p)
# this is stationary

In [ ]:
plt.figure(figsize=(20,5))
plt.grid()
plot_acf(diff, ax=plt.gca(), lags=30)
plt.show()
# ACF plot is not stationary

In [ ]:
#Second order non-seasonal differencing
diff2 = df['Yearly_revenue'].diff().diff().dropna()
plt.figure(figsize=(14,3))
plt.grid()
plt.plot(diff)
plt.show()

In [ ]:
print('Results of the Dickey Fuller Test:')
dftest = adfuller(diff2)
dfoutput=pd.Series(dftest[0:4],index=["Test Statistic",'P-value','#Lags used','Number of Observations'])
for key,value in dftest[4].items():
    dfoutput['Critical Value(%s)' %key] =value
print(dfoutput)
#this is not stationary

In [ ]:
kp = kpss(diff2)
p = kp[1]
print ("p-value for KPSS test (untransformed) = ", p)
# this is stationary

In [ ]:
plt.figure(figsize=(20,5))
plt.grid()
plot_acf(diff2, ax=plt.gca(), lags=30)
plt.show()
#not stationary

In [ ]:
#Third Order Non-Seasonal differencing
diff3 = df['Yearly_revenue'].diff().diff().diff().dropna()
plt.figure(figsize=(14,3))
plt.grid()
plt.plot(diff)
plt.show()

In [ ]:
print('Results of the Dickey Fuller Test:')
dftest = adfuller(diff3)
dfoutput=pd.Series(dftest[0:4],index=["Test Statistic",'P-value','#Lags used','Number of Observations'])
for key,value in dftest[4].items():
    dfoutput['Critical Value(%s)' %key] =value
print(dfoutput)
#this is stationary

In [ ]:
kp = kpss(diff2)
p = kp[1]
print ("p-value for KPSS test (untransformed) = ", p)
#thi is stationary

In [ ]:
plt.figure(figsize=(20,5))
plt.grid()
plot_acf(diff3, ax=plt.gca(), lags=30)
plt.show()

In [ ]:
#Arima
!pip install pmdarima

In [ ]:
!pip install pymannkendall pmdarima

In [ ]:
import statsmodels.api as sm
model_1 =sm.tsa.statespace.SARIMAX(train_df['Yearly_revenue'], order=(1,3,1),seasonal_order=(0,0,0,0))
model_1_fit = model_1.fit()

print(model_1_fit.summary())

In [ ]:
from pmdarima.arima import auto_arima
model_auto_arima = auto_arima(df['Yearly_revenue'], seasonal = False, trace = True, m = 1)
model_auto_arima = model_auto_arima.fit(df['Yearly_revenue'])

In [ ]:
model_2 =sm.tsa.statespace.SARIMAX(train_df['Yearly_revenue'], order=(0,1,0),seasonal_order=(0,0,0,0))
model_2_fit = model_2.fit()
print(model_2_fit.summary())

In [ ]:
test_prediction1 = model_1_fit.forecast(len(test_df))
forcast_further1 = model_1_fit.forecast(len(test_df['Yearly_revenue'])+10)
forcast_further1 = forcast_further1[test_df.index.max():]
forcast_further1

In [ ]:
test_prediction2 = model_2_fit.forecast(len(test_df))
forcast_further2 = model_2_fit.forecast(len(test_df['Yearly_revenue'])+10)
forcast_further2 = forcast_further2[test_df.index.max():]
forcast_further2

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error
mape_test1 = mean_absolute_percentage_error(test_df['Yearly_revenue'],test_prediction1)
print(mape_test1)

In [ ]:
mape_test2 = mean_absolute_percentage_error(test_df['Yearly_revenue'],test_prediction2)
print(mape_test2)

In [ ]:
model_1_fit.plot_diagnostics(figsize=(15,12))
plt.show

In [ ]:
model_2_fit.plot_diagnostics(figsize=(15,12))
plt.show

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(train_df['Yearly_revenue'],label='Actual')
plt.plot(test_df['Yearly_revenue'],label='Test')
plt.plot(test_prediction1,label='Predications')
plt.plot(forcast_further1,label='Future')
plt.title('SARIMAX Model Forecast')
plt.xlabel('Year')
plt.ylabel('Yearly_revenue')
plt.legend()
plt.show()